<a href="https://colab.research.google.com/github/kaynem25/Spark_tutorial/blob/main/challenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!apt-get update -qq > /dev/null
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!rm -f spark-3.1.1-bin-hadoop3.2.tgz spark-3.1.1-bin-hadoop3.2.tgz.1
!rm -rf /content/spark-3.1.1-bin-hadoop3.2
!wget https://archive.apache.org/dist/spark/spark-3.1.1/spark-3.1.1-bin-hadoop3.2.tgz
!ls -l spark-3.1.1-bin-hadoop3.2.tgz
!tar xf spark-3.1.1-bin-hadoop3.2.tgz
!pip install -q findspark
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.1.1-bin-hadoop3.2"
import findspark
findspark.init()
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()
spark

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
--2026-03-18 12:37:37--  https://archive.apache.org/dist/spark/spark-3.1.1/spark-3.1.1-bin-hadoop3.2.tgz
Resolving archive.apache.org (archive.apache.org)... 65.108.204.189, 2a01:4f9:1a:a084::2
Connecting to archive.apache.org (archive.apache.org)|65.108.204.189|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 228721937 (218M) [application/x-gzip]
Saving to: ‘spark-3.1.1-bin-hadoop3.2.tgz’

spark-3.1.1-bin-had 100%[===================>] 218.13M  10.6MB/s    in 93s     

2026-03-18 12:39:11 (2.34 MB/s) - ‘spark-3.1.1-bin-hadoop3.2.tgz’ saved [228721937/228721937]

-rw-r--r-- 1 root root 228721937 Feb 22  2021 spark-3.1.1-bin-hadoop3.2.tgz


In [8]:
mydata=spark.read.format("csv").option("header","true").load("original.csv")


In [9]:
mydata.show

<bound method DataFrame.show of DataFrame[id: string, first_name: string, last_name: string, gender: string, City: string, JobTitle: string, Salary: string, Latitude: string, Longitude: string]>

In [10]:
mydata.show()

+---+----------+----------+------+---------------+--------------------+---------+----------+-----------+
| id|first_name| last_name|gender|           City|            JobTitle|   Salary|  Latitude|  Longitude|
+---+----------+----------+------+---------------+--------------------+---------+----------+-----------+
|  1|   Melinde| Shilburne|Female|      Nowa Ruda| Assistant Professor|$57438.18|50.5774075| 16.4967184|
|  2|  Kimberly|Von Welden|Female|         Bulgan|       Programmer II|$62846.60|48.8231572|103.5218199|
|  3|    Alvera|  Di Boldi|Female|           null|                null|$57576.52|39.9947462|116.3397725|
|  4|   Shannon| O'Griffin|  Male|  Divnomorskoye|Budget/Accounting...|$61489.23|44.5047212| 38.1300171|
|  5|  Sherwood|   Macieja|  Male|      Mytishchi|            VP Sales|$63863.09|      null| 37.6489954|
|  6|     Maris|      Folk|Female|Kinsealy-Drinan|      Civil Engineer|$30101.16|53.4266145| -6.1644997|
|  7|     Masha|    Divers|Female|         Dachun|     

In [11]:
from pyspark.sql.functions import *

In [15]:
mydata2 = mydata.withColumn("clean_city", when(mydata['City'].isNull(), "unknown").otherwise(mydata['City']))

In [16]:
mydata2.show()

+---+----------+----------+------+---------------+--------------------+---------+----------+-----------+---------------+
| id|first_name| last_name|gender|           City|            JobTitle|   Salary|  Latitude|  Longitude|     clean_city|
+---+----------+----------+------+---------------+--------------------+---------+----------+-----------+---------------+
|  1|   Melinde| Shilburne|Female|      Nowa Ruda| Assistant Professor|$57438.18|50.5774075| 16.4967184|      Nowa Ruda|
|  2|  Kimberly|Von Welden|Female|         Bulgan|       Programmer II|$62846.60|48.8231572|103.5218199|         Bulgan|
|  3|    Alvera|  Di Boldi|Female|           null|                null|$57576.52|39.9947462|116.3397725|        unknown|
|  4|   Shannon| O'Griffin|  Male|  Divnomorskoye|Budget/Accounting...|$61489.23|44.5047212| 38.1300171|  Divnomorskoye|
|  5|  Sherwood|   Macieja|  Male|      Mytishchi|            VP Sales|$63863.09|      null| 37.6489954|      Mytishchi|
|  6|     Maris|      Folk|Femal

In [17]:
mydata2=mydata2.filter(mydata2['JobTitle'].isNotNull())

In [18]:
mydata2.show()

+---+----------+----------+------+---------------+--------------------+---------+----------+-----------+---------------+
| id|first_name| last_name|gender|           City|            JobTitle|   Salary|  Latitude|  Longitude|     clean_city|
+---+----------+----------+------+---------------+--------------------+---------+----------+-----------+---------------+
|  1|   Melinde| Shilburne|Female|      Nowa Ruda| Assistant Professor|$57438.18|50.5774075| 16.4967184|      Nowa Ruda|
|  2|  Kimberly|Von Welden|Female|         Bulgan|       Programmer II|$62846.60|48.8231572|103.5218199|         Bulgan|
|  4|   Shannon| O'Griffin|  Male|  Divnomorskoye|Budget/Accounting...|$61489.23|44.5047212| 38.1300171|  Divnomorskoye|
|  5|  Sherwood|   Macieja|  Male|      Mytishchi|            VP Sales|$63863.09|      null| 37.6489954|      Mytishchi|
|  6|     Maris|      Folk|Female|Kinsealy-Drinan|      Civil Engineer|$30101.16|53.4266145| -6.1644997|Kinsealy-Drinan|
|  8|   Goddart|     Flear|  Mal

In [19]:
mydata2=mydata2.withColumn("Clean_Salary", mydata2['Salary'].substr(2,100).cast("float"))

In [20]:
mydata2.show()

+---+----------+----------+------+---------------+--------------------+---------+----------+-----------+---------------+------------+
| id|first_name| last_name|gender|           City|            JobTitle|   Salary|  Latitude|  Longitude|     clean_city|Clean_Salary|
+---+----------+----------+------+---------------+--------------------+---------+----------+-----------+---------------+------------+
|  1|   Melinde| Shilburne|Female|      Nowa Ruda| Assistant Professor|$57438.18|50.5774075| 16.4967184|      Nowa Ruda|    57438.18|
|  2|  Kimberly|Von Welden|Female|         Bulgan|       Programmer II|$62846.60|48.8231572|103.5218199|         Bulgan|     62846.6|
|  4|   Shannon| O'Griffin|  Male|  Divnomorskoye|Budget/Accounting...|$61489.23|44.5047212| 38.1300171|  Divnomorskoye|    61489.23|
|  5|  Sherwood|   Macieja|  Male|      Mytishchi|            VP Sales|$63863.09|      null| 37.6489954|      Mytishchi|    63863.09|
|  6|     Maris|      Folk|Female|Kinsealy-Drinan|      Civil 

In [21]:
mean=mydata2.groupby().avg("Clean_Salary").take(1)[0][0]

In [22]:
print(mean)

55516.32088199837


In [24]:
mydata2 = mydata2.withColumn("New_salary", when(mydata2['Clean_Salary'].isNull(), lit(mean)).otherwise(mydata2['Clean_Salary']))

In [25]:
mydata2.show()

+---+----------+----------+------+---------------+--------------------+---------+----------+-----------+---------------+------------+----------------+
| id|first_name| last_name|gender|           City|            JobTitle|   Salary|  Latitude|  Longitude|     clean_city|Clean_Salary|      New_salary|
+---+----------+----------+------+---------------+--------------------+---------+----------+-----------+---------------+------------+----------------+
|  1|   Melinde| Shilburne|Female|      Nowa Ruda| Assistant Professor|$57438.18|50.5774075| 16.4967184|      Nowa Ruda|    57438.18|   57438.1796875|
|  2|  Kimberly|Von Welden|Female|         Bulgan|       Programmer II|$62846.60|48.8231572|103.5218199|         Bulgan|     62846.6|   62846.6015625|
|  4|   Shannon| O'Griffin|  Male|  Divnomorskoye|Budget/Accounting...|$61489.23|44.5047212| 38.1300171|  Divnomorskoye|    61489.23|  61489.23046875|
|  5|  Sherwood|   Macieja|  Male|      Mytishchi|            VP Sales|$63863.09|      null| 3

In [26]:
import numpy as np

In [27]:
latitudes = mydata2.select('Latitude')

In [28]:
latitudes = latitudes.filter(latitudes['Latitude'].isNotNull())

In [29]:
latitudes = latitudes.withColumn("Latitude_2", latitudes['Latitude'].cast("float")).select('Latitude_2')

In [30]:
latitudes.show()

+----------+
|Latitude_2|
+----------+
| 50.577408|
|  48.82316|
| 44.504723|
| 53.426613|
| 45.190517|
| 32.027935|
|  4.272793|
|     -5.85|
|  39.17238|
|  49.81518|
|  42.10148|
|  49.79233|
| 43.494576|
| 52.744167|
| 38.696247|
|-7.7232566|
| 40.717205|
|  49.16291|
| 40.757683|
|  48.49028|
+----------+
only showing top 20 rows



In [31]:
median = np.median(latitudes.collect())

In [32]:
print(median)

31.93397331237793


In [38]:
mydata2 = mydata2.withColumn("Latitude_3", when(mydata2['Latitude'].isNull(), lit(float(median))).otherwise(mydata2['Latitude']))

In [35]:
latitudes.show()

+----------+------------------+
|Latitude_2|        Latitude_3|
+----------+------------------+
| 50.577408| 50.57740783691406|
|  48.82316|48.823158264160156|
| 44.504723|44.504722595214844|
| 53.426613|53.426612854003906|
| 45.190517| 45.19051742553711|
| 32.027935| 32.02793502807617|
|  4.272793| 4.272792816162109|
|     -5.85|-5.849999904632568|
|  39.17238| 39.17237854003906|
|  49.81518|49.815181732177734|
|  42.10148|42.101478576660156|
|  49.79233| 49.79233169555664|
| 43.494576| 43.49457550048828|
| 52.744167| 52.74416732788086|
| 38.696247| 38.69624710083008|
|-7.7232566|-7.723256587982178|
| 40.717205| 40.71720504760742|
|  49.16291| 49.16291046142578|
| 40.757683| 40.75768280029297|
|  48.49028| 48.49028015136719|
+----------+------------------+
only showing top 20 rows



In [39]:
mydata2.show()

+---+----------+----------+------+---------------+--------------------+---------+----------+-----------+---------------+------------+----------------+-----------------+
| id|first_name| last_name|gender|           City|            JobTitle|   Salary|  Latitude|  Longitude|     clean_city|Clean_Salary|      New_salary|       Latitude_3|
+---+----------+----------+------+---------------+--------------------+---------+----------+-----------+---------------+------------+----------------+-----------------+
|  1|   Melinde| Shilburne|Female|      Nowa Ruda| Assistant Professor|$57438.18|50.5774075| 16.4967184|      Nowa Ruda|    57438.18|   57438.1796875|       50.5774075|
|  2|  Kimberly|Von Welden|Female|         Bulgan|       Programmer II|$62846.60|48.8231572|103.5218199|         Bulgan|     62846.6|   62846.6015625|       48.8231572|
|  4|   Shannon| O'Griffin|  Male|  Divnomorskoye|Budget/Accounting...|$61489.23|44.5047212| 38.1300171|  Divnomorskoye|    61489.23|  61489.23046875|     

In [1]:
mydata2.show()

NameError: name 'mydata2' is not defined

In [2]:
mydata2=mydata2.filter(mydata2['JobTitle'].isNotNull())

mydata2=mydata2.withColumn("Clean_Salary", mydata2['Salary'].substr(2,100).cast("float"))

mean=mydata2.groupby().avg("Clean_Salary").take(1)[0][0]

from pyspark.sql.functions import lit
mydata2 = mydata2.withColumn("New_salary", when(mydata2['Clean_Salary'].isNull(), lit(mean)).otherwise(mydata2['Clean_Salary']))

latitudes = mydata2.select('Latitude')

latitudes = latitudes.filter(latitudes['Latitude'].isNotNull())

latitudes = latitudes.withColumn("Latitude_2", latitudes['Latitude'].cast("float")).select('Latitude_2')

median = np.median(latitudes.collect())

mydata2 = mydata2.withColumn("Latitude_3", when(mydata2['Latitude'].isNull(), lit(float(median))).otherwise(mydata2['Latitude']))

NameError: name 'mydata2' is not defined

In [3]:
mydata2 = mydata.withColumn("clean_city", when(mydata['City'].isNull(), "unknown").otherwise(mydata['City']))

mydata2=mydata2.filter(mydata2['JobTitle'].isNotNull())

mydata2=mydata2.withColumn("Clean_Salary", mydata2['Salary'].substr(2,100).cast("float"))

mean=mydata2.groupby().avg("Clean_Salary").take(1)[0][0]

from pyspark.sql.functions import lit
mydata2 = mydata2.withColumn("New_salary", when(mydata2['Clean_Salary'].isNull(), lit(mean)).otherwise(mydata2['Clean_Salary']))

latitudes = mydata2.select('Latitude')

latitudes = latitudes.filter(latitudes['Latitude'].isNotNull())

latitudes = latitudes.withColumn("Latitude_2", latitudes['Latitude'].cast("float")).select('Latitude_2')

median = np.median(latitudes.collect())

mydata2 = mydata2.withColumn("Latitude_3", when(mydata2['Latitude'].isNull(), lit(float(median))).otherwise(mydata2['Latitude']))

NameError: name 'mydata' is not defined

In [6]:
mydata = spark.read.format("csv").option("header", "true").load("original.csv")

In [5]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q http://archive.apache.org/dist/spark/spark-3.1.1/spark-3.1.1-bin-hadoop3.2.tgz
!tar xf spark-3.1.1-bin-hadoop3.2.tgz
!pip install -q findspark
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.1.1-bin-hadoop3.2"
import findspark
findspark.init()
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()
spark

In [7]:
mydata.show()

+---+----------+----------+------+---------------+--------------------+---------+----------+-----------+
| id|first_name| last_name|gender|           City|            JobTitle|   Salary|  Latitude|  Longitude|
+---+----------+----------+------+---------------+--------------------+---------+----------+-----------+
|  1|   Melinde| Shilburne|Female|      Nowa Ruda| Assistant Professor|$57438.18|50.5774075| 16.4967184|
|  2|  Kimberly|Von Welden|Female|         Bulgan|       Programmer II|$62846.60|48.8231572|103.5218199|
|  3|    Alvera|  Di Boldi|Female|           null|                null|$57576.52|39.9947462|116.3397725|
|  4|   Shannon| O'Griffin|  Male|  Divnomorskoye|Budget/Accounting...|$61489.23|44.5047212| 38.1300171|
|  5|  Sherwood|   Macieja|  Male|      Mytishchi|            VP Sales|$63863.09|      null| 37.6489954|
|  6|     Maris|      Folk|Female|Kinsealy-Drinan|      Civil Engineer|$30101.16|53.4266145| -6.1644997|
|  7|     Masha|    Divers|Female|         Dachun|     

In [9]:
from pyspark.sql.functions import *
import numpy as np

mydata2 = mydata.withColumn("clean_city", when(mydata['City'].isNull(), "unknown").otherwise(mydata['City']))

mydata2=mydata2.filter(mydata2['JobTitle'].isNotNull())

mydata2=mydata2.withColumn("Clean_Salary", mydata2['Salary'].substr(2,100).cast("float"))

mean=mydata2.groupby().avg("Clean_Salary").take(1)[0][0]

mydata2 = mydata2.withColumn("New_salary", when(mydata2['Clean_Salary'].isNull(), lit(mean)).otherwise(mydata2['Clean_Salary']))

latitudes = mydata2.select('Latitude')

latitudes = latitudes.filter(latitudes['Latitude'].isNotNull())

latitudes = latitudes.withColumn("Latitude_2", latitudes['Latitude'].cast("float")).select('Latitude_2')

median = np.median(latitudes.collect())

mydata2 = mydata2.withColumn("Latitude_3", when(mydata2['Latitude'].isNull(), lit(float(median))).otherwise(mydata2['Latitude']))

In [10]:
mydata2.show()

+---+----------+----------+------+---------------+--------------------+---------+----------+-----------+---------------+------------+----------------+-----------------+
| id|first_name| last_name|gender|           City|            JobTitle|   Salary|  Latitude|  Longitude|     clean_city|Clean_Salary|      New_salary|       Latitude_3|
+---+----------+----------+------+---------------+--------------------+---------+----------+-----------+---------------+------------+----------------+-----------------+
|  1|   Melinde| Shilburne|Female|      Nowa Ruda| Assistant Professor|$57438.18|50.5774075| 16.4967184|      Nowa Ruda|    57438.18|   57438.1796875|       50.5774075|
|  2|  Kimberly|Von Welden|Female|         Bulgan|       Programmer II|$62846.60|48.8231572|103.5218199|         Bulgan|     62846.6|   62846.6015625|       48.8231572|
|  4|   Shannon| O'Griffin|  Male|  Divnomorskoye|Budget/Accounting...|$61489.23|44.5047212| 38.1300171|  Divnomorskoye|    61489.23|  61489.23046875|     

In [11]:
import pyspark.sql.functions as sqlfunc

In [12]:
genders = mydata2.groupBy("Gender")

In [13]:
genders.show()

AttributeError: 'GroupedData' object has no attribute 'show'

In [14]:
genders = mydata2.groupBy("Gender").agg(sqlfunc.avg("New_salary").alias("Avg_Salary"))

In [15]:
genders.show()

+------+------------------+
|Gender|        Avg_Salary|
+------+------------------+
|Female|55677.250125558036|
|  Male| 55361.09385573019|
+------+------------------+



In [17]:
df = mydata2.withColumn("Female_salary", when(mydata2['Gender'] == "F", mydata2['New_salary']).otherwise(lit(0)))

In [18]:
df = df.withColumn("Male_salary", when(df['Gender'] == "M", df['New_salary']).otherwise(lit(0)))

In [19]:
df.show()

+---+----------+----------+------+---------------+--------------------+---------+----------+-----------+---------------+------------+----------------+-----------------+-------------+-----------+
| id|first_name| last_name|gender|           City|            JobTitle|   Salary|  Latitude|  Longitude|     clean_city|Clean_Salary|      New_salary|       Latitude_3|Female_salary|Male_salary|
+---+----------+----------+------+---------------+--------------------+---------+----------+-----------+---------------+------------+----------------+-----------------+-------------+-----------+
|  1|   Melinde| Shilburne|Female|      Nowa Ruda| Assistant Professor|$57438.18|50.5774075| 16.4967184|      Nowa Ruda|    57438.18|   57438.1796875|       50.5774075|          0.0|        0.0|
|  2|  Kimberly|Von Welden|Female|         Bulgan|       Programmer II|$62846.60|48.8231572|103.5218199|         Bulgan|     62846.6|   62846.6015625|       48.8231572|          0.0|        0.0|
|  4|   Shannon| O'Griffi

In [20]:
df = mydata2.withColumn("Female_salary", when(mydata2['Gender'] == "Female", mydata2['New_salary']).otherwise(lit(0)))
df = df.withColumn("Male_salary", when(df['Gender'] == "Male", df['New_salary']).otherwise(lit(0)))

In [21]:
df.show()

+---+----------+----------+------+---------------+--------------------+---------+----------+-----------+---------------+------------+----------------+-----------------+----------------+----------------+
| id|first_name| last_name|gender|           City|            JobTitle|   Salary|  Latitude|  Longitude|     clean_city|Clean_Salary|      New_salary|       Latitude_3|   Female_salary|     Male_salary|
+---+----------+----------+------+---------------+--------------------+---------+----------+-----------+---------------+------------+----------------+-----------------+----------------+----------------+
|  1|   Melinde| Shilburne|Female|      Nowa Ruda| Assistant Professor|$57438.18|50.5774075| 16.4967184|      Nowa Ruda|    57438.18|   57438.1796875|       50.5774075|   57438.1796875|             0.0|
|  2|  Kimberly|Von Welden|Female|         Bulgan|       Programmer II|$62846.60|48.8231572|103.5218199|         Bulgan|     62846.6|   62846.6015625|       48.8231572|   62846.6015625|   

In [22]:
df = df.groupBy("JobTitle").agg(sqlfunc.avg("Female_salary").alias("Final_Female_salary"), sqlfunc.avg("Male_salary").alias("Final_Male_salary"))

In [23]:
df.show()

+--------------------+-------------------+------------------+
|            JobTitle|Final_Female_salary| Final_Male_salary|
+--------------------+-------------------+------------------+
|Systems Administr...|    50590.474609375|  15540.9501953125|
|   Media Manager III| 29586.436197916668|17381.920572916668|
|  Recruiting Manager| 34848.452473958336|  26383.4951171875|
|       Geologist III|       31749.046875|    12830.75390625|
|        Geologist II|                0.0|   43293.865234375|
|Database Administ...|                0.0|     52018.4609375|
|   Financial Analyst|    23353.776953125|       39606.05625|
|  Analyst Programmer|   16406.1287109375|  21042.9634765625|
|Software Engineer II|                0.0|      74782.640625|
|       Accountant IV|    82732.248046875|               0.0|
|    Product Engineer|     41825.48359375|       20464.94375|
|Software Test Eng...|   32218.6083984375|   27122.462890625|
|Safety Technician...|                0.0|   29421.529296875|
|    Jun

In [25]:
df = df.withColumn("Delta",df.Final_Female_salary-df.Final_Male_salary)

In [26]:
df.show()

+--------------------+-------------------+------------------+-------------------+
|            JobTitle|Final_Female_salary| Final_Male_salary|              Delta|
+--------------------+-------------------+------------------+-------------------+
|Systems Administr...|    50590.474609375|  15540.9501953125|   35049.5244140625|
|   Media Manager III| 29586.436197916668|17381.920572916668|       12204.515625|
|  Recruiting Manager| 34848.452473958336|  26383.4951171875|  8464.957356770836|
|       Geologist III|       31749.046875|    12830.75390625|     18918.29296875|
|        Geologist II|                0.0|   43293.865234375|   -43293.865234375|
|Database Administ...|                0.0|     52018.4609375|     -52018.4609375|
|   Financial Analyst|    23353.776953125|       39606.05625|   -16252.279296875|
|  Analyst Programmer|   16406.1287109375|  21042.9634765625| -4636.834765625001|
|Software Engineer II|                0.0|      74782.640625|      -74782.640625|
|       Accounta

In [27]:
cityavg = mydata2.groupBy("City").agg(sqlfunc.avg("New_salary").alias("City_avg"))

In [28]:
cityavg.show()

+-----------------+----------------+
|             City|        City_avg|
+-----------------+----------------+
|        Sułkowice|  33432.98828125|
|          Klippan|     77039.46875|
|      Trollhättan|53311.6845703125|
|        Shinaihai|    39544.640625|
|         Hongzhou|  35707.30859375|
|         Cipinang| 11617.509765625|
| Viejo Daan Banua|         43927.5|
|         Tsiatsan| 18795.439453125|
|       San Andres|  52426.80078125|
|           Krasna|   72022.7890625|
|      Springfield|40697.3251953125|
|            Město|  27797.98046875|
|Chaloem Phra Kiat|  54840.19921875|
|          Tadotsu|  55595.30078125|
|   Hénin-Beaumont|        55082.75|
|          Kajaani|  20224.83984375|
|           Duozhu|    71416.859375|
|           Abéché|   93375.1796875|
|     Habingkloang|     56892.96875|
|         Malishka|   76783.4765625|
+-----------------+----------------+
only showing top 20 rows



In [29]:
cityavg = cityavg.sort("City_avg", ascending=False)

In [30]:
cityavg.show()

+-----------------+-------------+
|             City|     City_avg|
+-----------------+-------------+
|        Mesopotam|  99948.28125|
|       Zhongcheng| 99942.921875|
|           Caxias|99786.3984375|
|      Karangtawar|99638.9921875|
|        Itabaiana|  99502.15625|
|           Pasian|  99421.34375|
|           Webuye| 99368.546875|
|      Yuktae-dong| 99250.828125|
|           Zinder|  99222.84375|
|   Timiryazevskiy|   99142.9375|
|        Sawahbaru|99013.7109375|
|          Madimba|98737.8671875|
|         Huangshi|  98690.34375|
|          Gharyan|   98679.3125|
|         Yŏnan-ŭp| 98628.609375|
|     Wringinputih|98603.8203125|
|Monte da Boavista|  98586.71875|
|          Klukeng|98439.4921875|
|         Murmashi|  98226.15625|
|        Fox Creek|      98138.0|
+-----------------+-------------+
only showing top 20 rows



In [31]:
from pyspark.sql.functions import *
import numpy as np

In [32]:
df = spark.read.format("csv").option("header", "true").load("challenge.csv")

In [33]:
df.show()

+---------------+--------------+-----------------+----------+
|     ip_address|       Country|      Domain Name|Bytes_used|
+---------------+--------------+-----------------+----------+
|  52.81.192.172|         China| odnoklassniki.ru|       463|
| 119.239.207.13|         China|         youtu.be|        51|
|  68.69.217.210|         China|        adobe.com|        10|
|   7.191.21.223|      Bulgaria|     linkedin.com|       853|
|   211.13.10.68|     Indonesia|          hud.gov|        29|
|   239.80.21.97|      Suriname|       smh.com.au|       218|
|106.214.106.233|       Jamaica|    amazonaws.com|        95|
| 127.242.24.138|         China| surveymonkey.com|       123|
|     99.2.6.139|Czech Republic|     geocities.jp|       322|
|   237.54.11.63|         China|       amazon.com|        83|
| 252.141.157.25|         Japan|      cornell.edu|       374|
|185.220.128.248|       Belgium|       weebly.com|       389|
|   151.77.19.45|   Afghanistan|independent.co.uk|       282|
|  9.161

In [34]:
df = df.withColumn("Mexico", when(df['Country'] == "Mexico", "Yes").otherwise("No"))

In [35]:
df.show()

+---------------+--------------+-----------------+----------+------+
|     ip_address|       Country|      Domain Name|Bytes_used|Mexico|
+---------------+--------------+-----------------+----------+------+
|  52.81.192.172|         China| odnoklassniki.ru|       463|    No|
| 119.239.207.13|         China|         youtu.be|        51|    No|
|  68.69.217.210|         China|        adobe.com|        10|    No|
|   7.191.21.223|      Bulgaria|     linkedin.com|       853|    No|
|   211.13.10.68|     Indonesia|          hud.gov|        29|    No|
|   239.80.21.97|      Suriname|       smh.com.au|       218|    No|
|106.214.106.233|       Jamaica|    amazonaws.com|        95|    No|
| 127.242.24.138|         China| surveymonkey.com|       123|    No|
|     99.2.6.139|Czech Republic|     geocities.jp|       322|    No|
|   237.54.11.63|         China|       amazon.com|        83|    No|
| 252.141.157.25|         Japan|      cornell.edu|       374|    No|
|185.220.128.248|       Belgium|  

In [37]:
df = df.groupBy("Mexico").agg(sqlfunc.sum("Bytes_Used").alias("Total_Bytes"))

In [38]:
df.show()

+------+-----------+
|Mexico|Total_Bytes|
+------+-----------+
|    No|   508076.0|
|   Yes|     6293.0|
+------+-----------+



In [40]:
df = df.groupBy("Country").agg(sqlfunc.countDistinct("Domain Name").alias("Total_Devices"))

AnalysisException: cannot resolve '`Country`' given input columns: [Mexico, Total_Bytes];
'Aggregate ['Country], ['Country, 'count(distinct 'Domain Name) AS Total_Devices#686]
+- Aggregate [Mexico#628], [Mexico#628, sum(cast(Bytes_Used#602 as double)) AS Total_Bytes#665]
   +- Project [ip_address#599, Country#600, Domain Name#601, Bytes_used#602, CASE WHEN (Country#600 = Mexico) THEN Yes ELSE No END AS Mexico#628]
      +- Relation[ip_address#599,Country#600,Domain Name#601,Bytes_used#602] csv


In [41]:
df = spark.read.format("csv").option("header", "true").load("challenge.csv")

In [42]:
df = df.withColumn("Mexico", when(df['Country'] == "Mexico", "Yes").otherwise("No"))

In [46]:
df = spark.read.format("csv").option("header", "true").load("challenge.csv")
df = df.withColumn("Mexico", when(df['Country'] == "Mexico", "Yes").otherwise("No"))
df = df.groupBy("Country").agg(sqlfunc.countDistinct("Domain Name").alias("Total_Domains"))

In [44]:
df.show()

+-----------+-------------+
|    Country|Total_Devices|
+-----------+-------------+
|       Chad|            1|
|     Russia|           51|
|   Paraguay|            1|
|      Yemen|            1|
|     Sweden|           28|
|Philippines|           60|
|   Malaysia|            5|
|     Turkey|            1|
|     Malawi|            2|
|    Germany|            5|
|    Comoros|            1|
|Afghanistan|            5|
|     Rwanda|            1|
|      Sudan|            1|
|     France|           21|
|     Greece|            8|
|  Sri Lanka|            3|
|   Dominica|            1|
|  Argentina|           14|
|    Belgium|            1|
+-----------+-------------+
only showing top 20 rows



In [47]:
df.show()

+-----------+-------------+
|    Country|Total_Domains|
+-----------+-------------+
|       Chad|            1|
|     Russia|           51|
|   Paraguay|            1|
|      Yemen|            1|
|     Sweden|           28|
|Philippines|           60|
|   Malaysia|            5|
|     Turkey|            1|
|     Malawi|            2|
|    Germany|            5|
|    Comoros|            1|
|Afghanistan|            5|
|     Rwanda|            1|
|      Sudan|            1|
|     France|           21|
|     Greece|            8|
|  Sri Lanka|            3|
|   Dominica|            1|
|  Argentina|           14|
|    Belgium|            1|
+-----------+-------------+
only showing top 20 rows



In [48]:
df = df.sort("Total_Domains", ascending=False)

In [49]:
df.show()

+--------------+-------------+
|       Country|Total_Domains|
+--------------+-------------+
|         China|          149|
|     Indonesia|          103|
|   Philippines|           60|
|        Russia|           51|
|        Brazil|           33|
|        Poland|           29|
|        Sweden|           28|
|         Japan|           25|
|Czech Republic|           23|
|      Portugal|           23|
|        France|           21|
|          Peru|           18|
|      Colombia|           17|
| United States|           15|
|     Argentina|           14|
|        Mexico|           13|
|       Ukraine|           13|
|      Thailand|           12|
|       Nigeria|           11|
|        Canada|           11|
+--------------+-------------+
only showing top 20 rows

